# GraphGrasp — Training on Google Colab

**Run:** abl14 (ABL13 features + Euler radians + train z-score + AHG-style CAM)

| Feature group | Dims |
|---|---|
| Dong wrist-local XYZ (normalized) | 3 |
| AHG angles | 10 |
| AHG distances | 10 |
| Dong Euler angles (beta/gamma, zero-padded) | 2 |
| **Total per node** | **25** |

**CSV required:** `hograspnet_abl13.csv`  
**Model:** `GCN_CAM_AHG_64_64_64_64_128_128_128_256`

## Pre-flight checklist
- Runtime set to GPU: `Runtime > Change runtime type > T4 GPU`
- Repo synced to `MyDrive/AIST-hand/`
- `hograspnet_abl13.csv` uploaded to Drive: `MyDrive/AIST-hand/data/processed/hograspnet_abl13.csv`


## 0. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU found. Go to Runtime > Change runtime type and select T4 GPU.')

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

In [ ]:
import os

DRIVE_DATA_DIR   = '/content/drive/MyDrive/AIST-hand/data'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/AIST-hand/experiments'
GITHUB_TOKEN     = ''  # paste your token here
GITHUB_USER      = 'isapedraza'
REPO_NAME        = 'AIST-hand'

print(f'Data dir:   {DRIVE_DATA_DIR}')
print(f'Output dir: {DRIVE_OUTPUT_DIR}')

## 2b. Run config — edit this cell for each new run

In [ ]:
import os

# ============================================================
# abl14: ABL13 features + radians/z-score + AHG-style CAM depth/channels
# ABL14 features per node: 25  (xyz_norm=3, ahg_angles=10, ahg_distances=10, euler=2)
# CSV/cache source: hograspnet_abl13.csv and ABL13 caches; ABL14 changes preprocessing/model only.
# ============================================================
RUN_NAME      = 'run_abl14_dong_xyz_norm_ahg_euler_rad_zscore_ahgcam'
NETWORK_TYPE  = 'GCN_CAM_AHG_64_64_64_64_128_128_128_256'
COLLAPSE      = 'none'
JOINT_ANGLES  = False
BONE_VECTORS  = False
VELOCITY      = False
MANO_POSE     = False
GLOBAL_SWING  = False
AHG_ANGLES    = True    # AHG angles (10 feat)
AHG_DISTANCES = True    # AHG distances (10 feat)
DONG_QUATS    = False   # no quaternions in abl13
DONG_EULER    = True    # Dong Euler angles (2 feat, zero-padded)
ADD_XYZ       = True    # XYZ coordinates (3 feat) -- Dong wrist-local
NORMALIZE_XYZ = True    # divide XYZ by ||MCP_middle - wrist|| on-the-fly
EULER_TO_RADIANS = True # convert Dong Euler degrees -> radians after cache load
FEATURE_ZSCORE   = True # z-score all node features with train-set stats
DONG_CSV_PATH = '/content/drive/MyDrive/AIST-hand/data/processed/hograspnet_abl13.csv'
NUM_EPOCHS    = 60
PATIENCE      = 15
LR_SCHEDULER        = 'plateau'
LR_PLATEAU_FACTOR   = 0.5
LR_PLATEAU_PATIENCE = 5
RUN_DIR_NAME  = 'run_abl14'
# ============================================================

CHECKPOINT = f'best_model_{RUN_NAME}.pth'

os.environ['GG_RUN_NAME']             = RUN_NAME
os.environ['GG_COLLAPSE']            = COLLAPSE
os.environ['GG_CHECKPOINT']          = CHECKPOINT
os.environ['GG_NETWORK_TYPE']        = NETWORK_TYPE
os.environ['GG_JOINT_ANGLES']        = str(JOINT_ANGLES).lower()
os.environ['GG_BONE_VECTORS']        = str(BONE_VECTORS).lower()
os.environ['GG_VELOCITY']            = str(VELOCITY).lower()
os.environ['GG_MANO_POSE']           = str(MANO_POSE).lower()
os.environ['GG_GLOBAL_SWING']        = str(GLOBAL_SWING).lower()
os.environ['GG_AHG_ANGLES']          = str(AHG_ANGLES).lower()
os.environ['GG_AHG_DISTANCES']       = str(AHG_DISTANCES).lower()
os.environ['GG_DONG_QUATS']          = str(DONG_QUATS).lower()
os.environ['GG_DONG_EULER']          = str(DONG_EULER).lower()
os.environ['GG_ADD_XYZ']             = str(ADD_XYZ).lower()
os.environ['GG_NORMALIZE_XYZ']       = str(NORMALIZE_XYZ).lower()
os.environ['GG_EULER_TO_RADIANS']    = str(EULER_TO_RADIANS).lower()
os.environ['GG_FEATURE_ZSCORE']      = str(FEATURE_ZSCORE).lower()
os.environ['GG_DONG_CSV_PATH']       = DONG_CSV_PATH
os.environ['GG_NUM_EPOCHS']          = str(NUM_EPOCHS)
os.environ['GG_PATIENCE']            = str(PATIENCE)
os.environ['GG_LR_SCHEDULER']        = LR_SCHEDULER
os.environ['GG_LR_PLATEAU_FACTOR']   = str(LR_PLATEAU_FACTOR)
os.environ['GG_LR_PLATEAU_PATIENCE'] = str(LR_PLATEAU_PATIENCE)

print(f'Run:           {RUN_NAME}')
print(f'Network:       {NETWORK_TYPE}')
print(f'Collapse:      {COLLAPSE}')
print(f'Features/node: 25  (xyz_norm=3, ahg_angles=10, ahg_distances=10, dong_euler=2)')
print(f'  add_xyz:       {ADD_XYZ}')
print(f'  normalize_xyz: {NORMALIZE_XYZ}')
print(f'  dong_quats:    {DONG_QUATS}')
print(f'  dong_euler:    {DONG_EULER}')
print(f'  euler_radians: {EULER_TO_RADIANS}')
print(f'  feature_zscore:{FEATURE_ZSCORE}')
print(f'  ahg_angles:    {AHG_ANGLES}')
print(f'  ahg_distances: {AHG_DISTANCES}')
print(f'CSV:           {DONG_CSV_PATH}')
print(f'Epochs:        {NUM_EPOCHS}  |  Patience: {PATIENCE}')
print(f'LR scheduler:  {LR_SCHEDULER} (factor={LR_PLATEAU_FACTOR}, patience={LR_PLATEAU_PATIENCE})')
print(f'Checkpoint:    {CHECKPOINT}')

exists  = os.path.exists(DONG_CSV_PATH)
size_mb = os.path.getsize(DONG_CSV_PATH) / 1e6 if exists else 0
status  = 'OK' if exists else 'MISSING'
print(f'\n  {status:8s} hograspnet_abl13.csv ({size_mb:.0f} MB)')
if not exists:
    raise FileNotFoundError(f'hograspnet_abl13.csv not found at {DONG_CSV_PATH}')


## 3. Clone the repository

In [ ]:
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    clone_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    os.system(f'git clone {clone_url} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull')
    print('Already cloned, pulled latest.')

## 4. Install dependencies

In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.version.cuda}')

!pip install -q torch-geometric
!pip install -q scikit-learn tensorboard pandas numpy
!pip install -q -e /content/AIST-hand/grasp-model

print('Done.')

## 5. Link data from Drive

In [ ]:
processed_dir = f'{REPO_DIR}/grasp-model/data/processed'
os.makedirs(processed_dir, exist_ok=True)

# abl13 CSV read directly from Drive via DONG_CSV_PATH — no copy needed.
print(f'  processed_dir: {processed_dir}')
print(f'  abl13 CSV:     {DONG_CSV_PATH} (read directly from Drive)')

## 5b. Restore processed graph cache from Drive (optional)

If you already ran this before, restore the `.pt` cache to skip the ~20-30 min graph conversion.

In [ ]:
import shutil
DRIVE_PROCESSED = f'{DRIVE_DATA_DIR}/processed'

if os.path.exists(DRIVE_PROCESSED):
    restored = 0
    for fname in os.listdir(DRIVE_PROCESSED):
        if not fname.endswith('.pt'):
            continue
        src = os.path.join(DRIVE_PROCESSED, fname)
        dst = os.path.join(processed_dir, fname)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            size_mb = os.path.getsize(dst) / 1e6
            print(f'  Restored: {fname} ({size_mb:.0f} MB)')
            restored += 1
    if restored == 0:
        print('  Nothing new to restore.')
else:
    print('  No cache on Drive yet -- will be generated during training (step 6).')

## 6. Train

First run converts the CSV to PyG graphs and caches `.pt` files (~20-30 min for 1.4M samples).
Subsequent runs load from cache.

In [ ]:
os.chdir(f'{REPO_DIR}/grasp-model')
print(f'cwd: {os.getcwd()}')

In [ ]:
!python train.py

## 7. Save model and TensorBoard logs to Drive

In [ ]:
import shutil
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

src = f'{REPO_DIR}/grasp-model/experiments/{CHECKPOINT}'
dst = os.path.join(DRIVE_OUTPUT_DIR, CHECKPOINT)
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f'Saved: {dst}')
else:
    print(f'Not found: {src}')

stats_name = f'{RUN_NAME}_feature_stats.pt'
stats_src = f'{REPO_DIR}/grasp-model/experiments/{stats_name}'
stats_dst = os.path.join(DRIVE_OUTPUT_DIR, stats_name)
if os.path.exists(stats_src):
    shutil.copy2(stats_src, stats_dst)
    print(f'Feature stats saved: {stats_dst}')

tb_src = f'{REPO_DIR}/grasp-model/experiments/runs'
tb_dst = os.path.join(DRIVE_OUTPUT_DIR, 'runs')
if os.path.exists(tb_src):
    if os.path.exists(tb_dst):
        shutil.rmtree(tb_dst)
    shutil.copytree(tb_src, tb_dst)
    print(f'TensorBoard logs saved: {tb_dst}')

## 8. TensorBoard (optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/AIST-hand/grasp-model/experiments/runs

## 9. Cache processed graphs to Drive (optional)

In [ ]:
DRIVE_PROCESSED = f'{DRIVE_DATA_DIR}/processed'
os.makedirs(DRIVE_PROCESSED, exist_ok=True)

local_processed = f'{REPO_DIR}/grasp-model/data/processed'
for fname in os.listdir(local_processed):
    if not fname.endswith('.pt'):
        continue
    src = os.path.join(local_processed, fname)
    dst = os.path.join(DRIVE_PROCESSED, fname)
    shutil.copy2(src, dst)
    size_mb = os.path.getsize(dst) / 1e6
    print(f'  {fname} ({size_mb:.0f} MB)')

## 10. Evaluation -- Plots & Metrics

Generates all analysis artifacts and saves them to Drive.

**Outputs saved to** `MyDrive/AIST-hand/experiments/run_abl14/`:
- `confusion_matrix_norm.png`
- `confusion_matrix_norm_gcn.csv`
- `per_class_f1.png`
- `classification_report.txt`
- `results.json`

In [ ]:
import os, sys, json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix, classification_report,
                             f1_score, precision_score, recall_score)
from torch_geometric.loader import DataLoader

REPO_DIR         = '/content/AIST-hand'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/AIST-hand/experiments'
MODEL_PATH       = os.path.join(DRIVE_OUTPUT_DIR, CHECKPOINT)
RUN_DIR          = os.path.join(DRIVE_OUTPUT_DIR, RUN_DIR_NAME)
os.makedirs(RUN_DIR, exist_ok=True)

os.chdir(f'{REPO_DIR}/grasp-model')
if f'{REPO_DIR}/grasp-model/src' not in sys.path:
    sys.path.insert(0, f'{REPO_DIR}/grasp-model/src')

from grasp_gcn.dataset.grasps import GraspsClass, GRASP_CLASS_NAMES
from grasp_gcn.network.utils import get_network

NETWORK_TYPE = globals().get('NETWORK_TYPE', 'GCN_CAM_AHG_64_64_64_64_128_128_128_256')
BATCH_SIZE   = 512
NUM_WORKERS  = 0
_collapse_arg = False if COLLAPSE == 'none' else COLLAPSE

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Run: {RUN_NAME}')
print(f'Network: {NETWORK_TYPE}')
print('Features/node: 25  (xyz_norm=3, ahg_angles=10, ahg_distances=10, dong_euler=2)')

dataset_test = GraspsClass(
    root='data/', split='test',
    collapse=_collapse_arg,
    add_joint_angles=JOINT_ANGLES,
    add_bone_vectors=BONE_VECTORS,
    add_velocity=VELOCITY,
    add_mano_pose=MANO_POSE,
    add_global_swing=GLOBAL_SWING,
    add_ahg_angles=AHG_ANGLES,
    add_ahg_distances=AHG_DISTANCES,
    add_dong_quats=DONG_QUATS,
    add_dong_euler=DONG_EULER,
    dong_csv_path=DONG_CSV_PATH,
    add_xyz=ADD_XYZ,
    normalize_xyz=NORMALIZE_XYZ,
)
if DONG_EULER and EULER_TO_RADIANS:
    dataset_test._data.x[:, -2:] = dataset_test._data.x[:, -2:] * (np.pi / 180.0)
if FEATURE_ZSCORE:
    stats_path = os.path.join(DRIVE_OUTPUT_DIR, f'{RUN_NAME}_feature_stats.pt')
    if not os.path.exists(stats_path):
        stats_path = f'{REPO_DIR}/grasp-model/experiments/{RUN_NAME}_feature_stats.pt'
    stats = torch.load(stats_path, map_location='cpu')
    dataset_test._data.x = (dataset_test._data.x.float() - stats['mean']) / stats['std']
    print(f'Applied feature stats: {stats_path}')

test_loader  = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
NUM_CLASSES  = dataset_test.num_classes
CLASS_NAMES  = [GRASP_CLASS_NAMES[i] for i in range(NUM_CLASSES)]
NUM_FEATURES = dataset_test.num_features
EXPECTED_FEATURES = 25

print(f'Test samples: {len(dataset_test)} | Classes: {NUM_CLASSES} | Features/node: {NUM_FEATURES}')
assert NUM_FEATURES == EXPECTED_FEATURES, f'Expected {EXPECTED_FEATURES} features/node, got {NUM_FEATURES}'

model = get_network(NETWORK_TYPE, NUM_FEATURES, NUM_CLASSES).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print(f'Model loaded from: {MODEL_PATH}')

y_true, y_pred = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        preds = model(batch).argmax(1).cpu().numpy()
        labels = batch.y.view(-1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(labels)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

acc      = (y_true == y_pred).mean()
macro_f1 = f1_score(y_true, y_pred, average='macro')
print(f'\nTest Accuracy : {acc:.4f}')
print(f'Macro F1      : {macro_f1:.4f}')

report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3)
print('\n' + report)
report_path = os.path.join(RUN_DIR, 'classification_report.txt')
with open(report_path, 'w') as f:
    f.write(f'{RUN_NAME} -- {NETWORK_TYPE}\n')
    f.write(f'Features/node: {NUM_FEATURES}  (xyz_norm=3, ahg_angles=10, ahg_distances=10, dong_euler=2)\n')
    f.write(f'Euler to radians: {EULER_TO_RADIANS} | Feature z-score: {FEATURE_ZSCORE}\n')
    f.write(f'Test Accuracy : {acc:.4f}\n')
    f.write(f'Macro F1      : {macro_f1:.4f}\n\n')
    f.write(report)
print(f'Saved: {report_path}')

cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title(f'Confusion Matrix -- {RUN_NAME}\nTest acc: {acc:.3f} | Macro F1: {macro_f1:.3f}', fontsize=11)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ticks = np.arange(NUM_CLASSES)
ax.set_xticks(ticks); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=7)
ax.set_yticks(ticks); ax.set_yticklabels(CLASS_NAMES, fontsize=7)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        val = cm_norm[i, j]
        if val > 0.05:
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=5, color='white' if val > 0.6 else 'black')
plt.tight_layout()
cm_png = os.path.join(RUN_DIR, 'confusion_matrix_norm.png')
fig.savefig(cm_png, dpi=150); plt.show()
print(f'Saved: {cm_png}')

cm_df = pd.DataFrame(cm_norm, index=CLASS_NAMES, columns=CLASS_NAMES)
cm_df.to_csv(os.path.join(RUN_DIR, 'confusion_matrix_norm_gcn.csv'))

f1_per_class   = f1_score(y_true, y_pred, average=None, labels=range(NUM_CLASSES))
prec_per_class = precision_score(y_true, y_pred, average=None, labels=range(NUM_CLASSES))
rec_per_class  = recall_score(y_true, y_pred, average=None, labels=range(NUM_CLASSES))
support        = np.bincount(y_true, minlength=NUM_CLASSES)

colors = ['#d62728' if f < 0.4 else '#ff7f0e' if f < 0.6 else '#2ca02c' for f in f1_per_class]
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(NUM_CLASSES)
bars = ax.bar(x, f1_per_class, color=colors, edgecolor='white', linewidth=0.5)
ax.axhline(macro_f1, color='steelblue', linestyle='--', linewidth=1.2, label=f'Macro F1 = {macro_f1:.3f}')
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('F1 score'); ax.set_title(f'Per-class F1 -- {RUN_NAME}'); ax.set_ylim(0, 1.1); ax.legend()
for bar, sup in zip(bars, support):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.01,
            f'{sup//1000}k', ha='center', va='bottom', fontsize=6, rotation=90)
plt.tight_layout()
fig.savefig(os.path.join(RUN_DIR, 'per_class_f1.png'), dpi=150); plt.show()

results = {
    'run': RUN_NAME,
    'model': NETWORK_TYPE,
    'collapse': COLLAPSE,
    'features_per_node': NUM_FEATURES,
    'add_xyz': ADD_XYZ,
    'normalize_xyz': NORMALIZE_XYZ,
    'euler_to_radians': EULER_TO_RADIANS,
    'feature_zscore': FEATURE_ZSCORE,
    'ahg_angles': AHG_ANGLES,
    'ahg_distances': AHG_DISTANCES,
    'dong_quats': DONG_QUATS,
    'dong_euler': DONG_EULER,
    'num_classes': NUM_CLASSES,
    'test_accuracy': round(float(acc), 4),
    'macro_f1': round(float(macro_f1), 4),
    'per_class': {
        CLASS_NAMES[i]: {
            'precision': round(float(prec_per_class[i]), 4),
            'recall':    round(float(rec_per_class[i]),  4),
            'f1':        round(float(f1_per_class[i]),   4),
            'support':   int(support[i]),
        } for i in range(NUM_CLASSES)
    },
}
results_path = os.path.join(RUN_DIR, 'results.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved: {results_path}')
print(f'\nAll artifacts saved to: {RUN_DIR}')